Load the data

In [12]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\YoungAge_combined_run1_400_noZ.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['run1_data']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

(72, 400, 400)
(400, 1, 72, 400)
Data shape: torch.Size([400, 1, 72, 400])


In [13]:
def check_zero_mean_unit_range(data, atol=1e-5):
    if hasattr(data, "detach"):
        arr = data.detach().cpu().numpy()
    else:
        arr = np.asarray(data)

    mean_val = arr.mean()
    min_val = arr.min()
    max_val = arr.max()

    is_zero_mean = np.isclose(mean_val, 0.0, atol=atol)
    has_minus_one = np.isclose(min_val, -1.0, atol=atol)
    has_plus_one = np.isclose(max_val, 1.0, atol=atol)

    print(f"Doing checks for zero mean and unit range:")
    print(f"mean: {mean_val:.6f}")
    print(f"standard deviation: {arr.std():.6f}")
    print(f"min:  {min_val:.6f}")
    print(f"max:  {max_val:.6f}")
    print(f"zero mean: {is_zero_mean}")
    print(f"min == -1: {has_minus_one}")
    print(f"max ==  1: {has_plus_one}")

    return is_zero_mean and has_minus_one and has_plus_one

# Example:
# check_zero_mean_unit_range(X)

def z_normalize(data, eps=1e-8):
    if hasattr(data, "detach"):
        mean = data.mean(dim=(1, 2, 3), keepdim=True)
        std = data.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp_min(eps)
        return (data - mean) / std

    arr = np.asarray(data)
    mean = arr.mean(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = arr.std(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = np.maximum(std, eps)
    return (arr - mean) / std


In [14]:
check_zero_mean_unit_range(X)
z_normalized_X = z_normalize(X)
check_zero_mean_unit_range(z_normalized_X)

Doing checks for zero mean and unit range:
mean: 0.266214
standard deviation: 0.275651
min:  -0.845308
max:  1.000000
zero mean: False
min == -1: False
max ==  1: True
Doing checks for zero mean and unit range:
mean: -0.000000
standard deviation: 1.000000
min:  -4.357945
max:  3.824302
zero mean: True
min == -1: False
max ==  1: False


False